In [11]:
print("HI")


HI


In [12]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Sarvam ASR Assignment: Diarization Benchmarking & Enhancement
---

**Deadline: Saturday, 22 August 2026, EOD**

You are given a CSV containing 100 YouTube videos with ground-truth diarization labels and reference transcripts. Your task is to parse these labels, benchmark existing diarization and ASR systems on them, and then **improve** the output.

### Input

`youtube_segments.csv` — one row per segment. The csv is present at https://drive.google.com/file/d/1Ijs1IWypIY2GAjpNUKV2XNZY6o7dFvSm/view?usp=sharing

Columns:

- `video_id` — YouTube video id
- `youtube_link` — source URL
- `start_sec`, `end_sec` — the window to extract. Use **only** `[start_sec, end_sec]`.
- `diarization_segments` — **ground-truth diarization**, formatted as `Speaker A [start-end] | Speaker B [start-end] | ...`
- `asr_segments` — **reference transcript** for the same segment, formatted as `[start-end] text | [start-end] text | ...`

`diarization_segments` and `asr_segments` share the **same segment boundaries in the same order** — the *n*-th `asr_segments` entry is the transcript of the *n*-th `diarization_segments` turn — so you can join speaker to text by index or by time. All timestamps are **relative to `start_sec`** (0 = start of the extracted clip).

If for any reason the ground-truth labels look incorrect, hand-label a subset and call out the limitation.

---

### Step 1 - Audio (or Video) Extraction

- Download the **audio** track from each YouTube link (yt-dlp / ffmpeg).
- Trim to `[start_sec, end_sec]`.
- Store as 16 kHz mono WAV.
- Optionally download the **video** track for the same window as well, if you plan to use visual cues in Step 4.

### Step 2 - Baseline Diarization Benchmarking

Run the extracted segments through top open-source diarization models.

Report standard metrics:

- **DER**
- **JER**
- Speaker-count accuracy

**Important:** Do NOT ignore overlapping speech regions when computing metrics. Overlap must be scored.

### Step 3 - ASR on Diarized Segments

Run ASR on the diarized speaker segments to produce speaker-attributed transcripts. Benchmark more than one STT system (e.g. Sarvam Saaras, Whisper, etc.) — the reference transcript in `asr_segments` is what you score against, not a substitute for running ASR yourself. Think about strategies for handling overlapping speech regions during ASR - how do you transcribe segments where multiple speakers are active simultaneously?

Also report:

- **cpWER**
- **WDER**

### Step 4 - Improving the Output

Take the best ASR + diarization combination you found while benchmarking, and build a pipeline on top of it that improves the output. Quantify the improvement over the same metrics as before. It is up to you whether you target diarization, ASR, or both.

**The ground truth is never an input to your pipeline.** `diarization_segments` and `asr_segments` are only for computing the final scores.

You are free to choose the strategy. Some directions:

- **Semantic / LLM-based correction** — feed the speaker-attributed text + timestamps to an LLM to detect and correct boundary errors, speaker confusion, and turn-merging issues. Any LLM is acceptable: paid APIs (GPT, Claude, Gemini, etc.) or open-source models. Prompting strategy, structured-output schema, chain-of-thought, and multi-pass refinement are yours to design.
- **Acoustic and visual cues** — use the video alongside the audio (e.g. active-speaker detection, face tracks, lip motion) to refine speaker boundaries and identities.
- Any combination of the above, or anything else you can justify.

**Hint:** Think about how transcript content can reveal diarization mistakes - e.g. unnaturally short segments, repeated/stuttered text across a speaker boundary suggesting a false split, or incoherent speaker transitions that indicate speaker confusion.

Also include:
- The papers / ideas your design draws from
- Multilingual / Indic-specific observations and failure modes

---

### Deliverables

Mail a Google Drive folder link (with all of the following):

1. A reproducible Colab notebook with the full pipeline end-to-end.
2. A results table: baseline vs. improved DER / JER / cpWER / WDER per model per video.
3. A short writeup (1–2 pages) covering approach, design choices, what worked, what didn't, failure cases, and references.

---

### Evaluation

- Correctness of pipeline and metrics
- Magnitude and consistency of improvement over baseline
- Depth of analysis and engineering judgement
- Clarity of code and writeup
- Breadth of techniques explored (bonus)


    

---

# Step 0 — Dataset: structure and observations

Everything below was measured from the real `youtube_segments.csv` (100 rows,
9,942 segments) before any code was written. These facts drive the design of
every later step, so they are recorded here rather than in a side document.

### Shape and keys

| Fact | Value | Why it matters |
|---|---|---|
| Rows × columns | 100 × 6, no nulls | — |
| Column order in file | `video_id, start_sec, end_sec, youtube_link, diarization_segments, asr_segments` | **Differs from the brief.** Index by name, never by position. |
| Unique `video_id` | **99**, not 100 | `RL2fhIEEbZg` appears **twice**, with the disjoint windows `0–65` and `66–1034`. The primary key is `(video_id, start_sec, end_sec)`, so every filename and cache key uses a composite `clip_id`. |
| `youtube_link` | always `https://www.youtube.com/watch?v={video_id}` | Redundant; `video_id` is the reliable field. |
| `video_id` charset | all match `^[A-Za-z0-9_-]{11}$`, 29 contain `-`/`_`, none *start* with one | No CLI arg-parsing hazard, but IDs are still passed as argv lists, never shell strings. |
| `start_sec` / `end_sec` | floats, always integral | start 0–677 (26 rows start at 0), end 60–2005. |

### Volume

Clip length is bimodal: min 50 s, median 325 s, max 1822 s — 38 rows are ≤120 s
while 27 rows exceed 600 s. **Total 12.28 h**, which is ≈1.41 GB as 16 kHz mono
s16 WAV. The full source videos total 29.3 h, so fetching only the requested
window saves ~58% of the bytes. No row's `end_sec` exceeds its source video's
duration, so every window is extractable in principle.

### Availability

Checked live. **99 of 99 videos are reachable except one**: `GUVrL5ltiP4` is
deleted (`Video unavailable`), so the ceiling for Step 1 is **99/100 rows**.

> `2HGP34TNvjg` returns HTTP 401 from YouTube's oEmbed endpoint, which looks
> like a dead video but only means *embedding is disabled* — yt-dlp downloads it
> normally. Availability is judged by an actual fetch, never by an oEmbed probe.

### Label format

Both label columns are `|`-separated, and splitting on `|` then anchoring a
timestamp regex at each piece parses **100% of 9,942 segments** with zero
failures. Two details make the naive approach wrong:

* **82 transcript segments contain literal `[` or `]`** — square-bracket
  code-switch glosses such as `டவுட் [doubt]`. A global `findall` over
  `\[...\]` mis-parses the file, so the regex is anchored at the start of each
  piece.
* `diarization_segments[n]` and `asr_segments[n]` carry **byte-identical
  timestamps for every single segment** (0 mismatches), so index-join and
  time-join are equivalent. Segments are already sorted by start time.

Speaker labels are `Speaker A`–`Speaker H`, always contiguous from A. Speakers
per clip — the reference for Step 2's speaker-count accuracy:

| Speakers | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|
| Clips | 26 | 29 | 28 | 9 | 4 | 2 | 2 |

### Timing quirks that change how metrics must be computed

* **Ground truth runs past the clip in 85 / 100 rows** — median **+1.43 s**,
  max **+6.41 s** (`Iqhnt8ENm3U`), uncorrelated with clip length (r = −0.11), so
  it is a systematic annotation-window offset rather than drift. Three rows go
  the other way, with long unannotated tails: `Cku_X_SL7qU` **−90.3 s**,
  `OxYCBQKZ3iY` −12.4 s, `CO_8ppdzq9U` −6.1 s.
  ⇒ Step 1 publishes exactly `[start_sec, end_sec]`, and Step 2 must **crop the
  reference to the UEM** rather than extend the audio.
* **Overlap is pervasive and must be scored** (the brief forbids ignoring it):
  **7.16% of corpus time has ≥2 distinct speakers active**, and only **9 of 100
  clips** have no overlap at all. `QuA_B6IZ6Ls` is 35% overlapped.
  *(Summing pairwise overlaps instead gives 8.49% — that double-counts 3-way
  overlap, so the profile uses a sweep-line over elementary intervals.)*
* **Turns are utterance-level, not turn-level**: 22.4% of adjacent segment pairs
  share a speaker label, so a merged/turn-level view must be compared too.
* **20% of segments are under 0.5 s** (1,953 of them; 3,473 are under 1 s),
  median segment 1.89 s. These are backchannels, and most diarizers miss them.

### Two corrupt ground-truth segments

Dropped by the parser with a flag, never silently:

| Clip | Segment | Problem | Text |
|---|---|---|---|
| `QuA_B6IZ6Ls` | 93 | `Speaker C [939.09-155.91]` — **end < start** (−783 s); neighbours sit at 931.5 and 947.0, so the intended end was ~945.9 | `<unintelligible>` |
| `6ZeRgvDHwcI` | 0 | `Speaker A [1.61-1.61]` — **zero duration** | `<noise>` |

### Content: multilingual and heavily code-switched

The corpus spans **nine Indic scripts**, so Step 3 cannot use a single fixed ASR
language code — the per-clip `lang_hint` column carries it forward:

| Script | Deva | Gujarati | Telugu | Tamil | Kannada | Odia | Bengali | Gurmukhi | Malayalam |
|---|---|---|---|---|---|---|---|---|---|
| Clips | 25 | 12 | 12 | 10 | 9 | 9 | 8 | 8 | 7 |

Devanagari covers both Hindi and Marathi videos and is labelled `hi_or_mr`
rather than guessed. **26.1% of all transcript characters are Latin** (median
video 27%, heaviest 48%) — code-switching is the norm, not an edge case.

**Non-speech tags**: `<unintelligible>` 264, `<noise>` 166, `<laughter>` 148,
`<vocalization>` 81, `<background_speech>` 12, `<uhhh>` 1. **404 segments (4.1%)
are tag-only or empty** — real diarization turns, but they must be excluded from
WER scoring.

**The code-switch gloss convention is messier than a single rule can handle.**
22,547 glosses across 4,624 segments (46%). A Step 3 normalizer must survive all
of these, or cpWER is measuring the convention rather than the ASR:

| Variant | Example | Count |
|---|---|---|
| No space before paren | `कॉफी(coffee)` | 2,879 segs |
| Space before paren | `ஹாய் (hi)` | 1,916 segs |
| **Square brackets** | `டவுட் [doubt]` | 521 glosses / 76 segs / 8 videos |
| **Curly braces** | `ट्राई {try}` | 10 glosses / 3 segs / 1 video |
| **Doubled / unbalanced parens** | `((Stardom))`, `ଫାସିଲିଟିଜ((facilities)` | 29 segs |
| **Native suffix re-attached after gloss** | `चॅनल(channel)-वर`, `ସେକ୍ସନରେ(section)-ରେ` | 1,831 |
| Numeral gloss | `एक(1)`, `दीड(1.5)` | 1,300 |
| Acronym gloss | `(CCBK)`, `(MTB)` | 331 |

A naive `re.sub(r'\([^)]*\)', '', text)` corrupts the 29 unbalanced segments,
leaves a dangling `-suffix` on 1,831 more, and misses the square- and
curly-bracket variants entirely — and **16 segments across 8 videos mix more
than one bracket style**, so the styles cannot be handled per-video either. Punctuation includes the Devanagari
danda `।` (1,706) alongside `.` (1,891), `,` (743) and `?` (531), so
normalization has to strip both ASCII and Indic punctuation.

---

# Step 1 — Audio extraction

Download each YouTube window, trim to `[start_sec, end_sec]`, publish as
**16 kHz mono WAV of exactly `round(duration × 16000)` samples**, checkpointed
to Google Drive with skip-if-exists.

**Architecture.** This notebook is orchestration only — config, stage toggles,
calls and display. All logic lives in the `sarvam_diar/` package beside it, so
it is reviewable outside the notebook and reusable across Steps 2–4.
`%autoreload` picks up edits to those modules without restarting the runtime.

**Two invariants the code is strict about:**

1. **Boundaries are exact.** Every published WAV holds exactly
   `round((end_sec - start_sec) * 16000)` samples, so each clip's timeline is
   exactly `[0, duration]` and the ground-truth timestamps — which are relative
   to `start_sec` — index straight into it with no offset arithmetic anywhere
   downstream. `DUR_TOL_SEC` is only a *diagnostic* that decides whether a fetch
   is trustworthy; it never defines the published boundary.
2. **A file existing proves nothing.** A Colab runtime can die mid-copy, so a
   clip counts as done only when its sidecar exists (written *last*, as the
   commit marker) **and** the WAV probes as 16 kHz mono `pcm_s16le` with the
   exact expected sample count.

## 1.0 — Setup

Installs `yt-dlp` (keep it current: YouTube extractor breakage is the single
most common cause of failure), mounts Drive, and puts `sarvam_diar/` on the
path. Everything here is cheap and safe to re-run.

In [13]:
# --- dependencies -----------------------------------------------------------
# ffmpeg/ffprobe are preinstalled on Colab. yt-dlp changes weekly, so upgrade it.
%pip install -q --upgrade yt-dlp
%pip install -q gdown

import shutil, subprocess, sys
from pathlib import Path

for tool in ("ffmpeg", "ffprobe", "yt-dlp", "git"):
    path = shutil.which(tool)
    print(f"{tool:9s} {path or 'MISSING'}")
    assert path, f"{tool} not found on PATH"

# --- Drive ------------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# --- code ------------------------------------------------------------------
# This notebook is orchestration only; the logic lives in the sarvam_diar
# package. On Colab it is cloned fresh from GitHub into local disk (not Drive --
# git over the Drive FUSE mount is slow and flaky). The repo is small, so a
# clone costs a second or two and guarantees the code matches the notebook.
REPO_URL = "https://github.com/ParvGoyal08/MultilingualASR.git"
CODE_DIR = Path("/content/sarvam-assignment")

def sync_code():
    """Clone or fast-forward the package. Returns the checkout path."""
    if (CODE_DIR / ".git").exists():
        r = subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only", "-q"],
                           capture_output=True, text=True)
        if r.returncode:
            # Local checkout diverged or is broken -- a fresh clone is always
            # correct here because nothing is ever edited on the Colab side.
            print("pull failed, re-cloning:", r.stderr.strip()[:200])
            shutil.rmtree(CODE_DIR, ignore_errors=True)
        else:
            return CODE_DIR
    subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE_DIR)], check=True)
    return CODE_DIR

if IN_COLAB:
    PROJECT_DIR = sync_code()
    sha = subprocess.run(["git", "-C", str(PROJECT_DIR), "log", "-1", "--format=%h  %s"],
                         capture_output=True, text=True).stdout.strip()
    print(f"\ncode @ {sha}")
else:
    # Running locally: use the folder this notebook sits in.
    PROJECT_DIR = next((p for p in (Path.cwd(), Path.cwd().parent)
                        if (p / "sarvam_diar" / "__init__.py").exists()), None)
    assert PROJECT_DIR, "sarvam_diar/ not found next to the notebook"

sys.path.insert(0, str(PROJECT_DIR))
print("PROJECT_DIR", PROJECT_DIR)

# Drop any previously-imported sarvam_diar modules so the freshly pulled code is
# what actually gets imported. This is the mechanism that makes re-running this
# cell pick up a new commit -- without it Python serves the old modules from
# sys.modules and a `git pull` appears to do nothing.
for _m in [k for k in list(sys.modules)
           if k == "sarvam_diar" or k.startswith("sarvam_diar.")]:
    del sys.modules[_m]

# autoreload is a convenience on top of that (it also catches edits made
# in-place). Colab's IPython ships a version whose autoreload does
# `from imp import reload`, and `imp` was removed in Python 3.12 -- so this is
# best-effort and the purge above is what we actually rely on.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("autoreload: on")
except Exception as exc:
    print(f"autoreload unavailable ({type(exc).__name__}) -- re-run this cell "
          "after a push to pick up new code")

import pandas as pd

import sarvam_diar
from sarvam_diar import config, data, extraction, reference, utils
print("sarvam_diar", sarvam_diar.__version__)

ffmpeg    /usr/bin/ffmpeg
ffprobe   /usr/bin/ffprobe
yt-dlp    /usr/local/bin/yt-dlp
git       /usr/bin/git
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

code @ e60cee9  Don't hard-depend on autoreload; purge modules explicitly instead
PROJECT_DIR /content/sarvam-assignment
autoreload unavailable (ModuleNotFoundError) -- re-run this cell after a push to pick up new code
sarvam_diar 0.1.0


## 1.1 — Config and stage flags

One visible cell holding every path and toggle. Each stage checkpoints to Drive,
so the normal workflow is: enable one stage, let it finish, flip it off, move
on. Re-running the notebook top-to-bottom then costs nothing but a few file
reads — which is what makes it survivable on a runtime that disconnects every
90 minutes.

In [14]:
from sarvam_diar.config import Config, StageFlags

# Durable storage. On Colab this defaults to Drive so it survives a runtime
# reset; work_dir is fast local scratch (ffmpeg never writes into the Drive
# FUSE mount -- it is slow and gives no write atomicity).
cfg = Config.create(
    root=None,             # None -> Drive on Colab, ./pipeline_out locally
    # force_client="android",   # pin a yt-dlp player client to skip the ladder search
    # cookies_file=None,        # or drop a cookies.txt in <root>/data/ for bot-gated videos
)

flags = StageFlags(
    run_extraction=True,       # Step 1
    build_reference=True,      # scoring reference (cheap, safe to leave on)
    run_diarization=False,     # Step 2 (not implemented yet)
    run_asr=False,             # Step 3 (not implemented yet)
    run_refinement=False,      # Step 4 (not implemented yet)

    force_redo=False,          # ignore checkpoints for the enabled stages
    retry_failed_only=False,   # only re-attempt non-permanent failures
    limit=None,                # e.g. 3 for a smoke run
    only_clip_ids=None,        # e.g. ["T3I2T-cfhG4"] to target specific clips
)

print(cfg.describe())
print(flags.describe())

{
  "root": "/content/drive/MyDrive/sarvam_diarization",
  "work_dir": "/content/work",
  "sample_rate": "16000",
  "dur_tol_sec": "0.25",
  "max_attempts": "4",
  "force_client": "None",
  "cookies_file": "None"
}
{
  "run_extraction": true,
  "build_reference": true,
  "run_diarization": false,
  "run_asr": false,
  "run_refinement": false,
  "force_redo": false,
  "retry_failed_only": false,
  "limit": null,
  "only_clip_ids": null
}


## 1.2 — Load the CSV and parse the ground truth

The CSV is fetched to Drive once and reused (skip-if-exists). Parsing asserts
that the two label columns agree segment-for-segment, and drops the two corrupt
segments documented in Step 0 with an explicit record of what was excluded.

In [15]:
df = data.load_segments_csv(cfg)
clips = data.parse_ground_truth(df)

print(f"{len(df)} rows, {df.video_id.nunique()} unique video_ids, "
      f"{sum(len(c.segments) for c in clips)} segments")

# The duplicated video_id is why clip_id is composite, not just video_id.
dupes = df[df.video_id.duplicated(keep=False)]
print("\nSame video, two windows -> two distinct clip_ids:")
display(dupes[["video_id", "start_sec", "end_sec", "clip_id"]])

gt = data.clips_to_frame(clips)
display(gt.head(3))

16:47:17 | INFO    | sarvam_diar | downloading segments CSV -> /content/drive/MyDrive/sarvam_diarization/data/youtube_segments.csv
16:47:21 | INFO    | sarvam_diar | parsed 100 clips, 9940 segments, 2 dropped as malformed, 0 unparsable entries
100 rows, 99 unique video_ids, 9940 segments

Same video, two windows -> two distinct clip_ids:


,video_id,start_sec,end_sec,clip_id
87,RL2fhIEEbZg,0.0,65.0,RL2fhIEEbZg__0_65
88,RL2fhIEEbZg,66.0,1034.0,RL2fhIEEbZg__66_1034


,clip_id,video_id,youtube_link,start_sec,end_sec,requested_dur_sec,n_gt_segments,n_gt_speakers,gt_speakers,gt_min_start,...,gt_speech_sec,gt_speaker_time_sec,gt_overlap_sec,gt_overlap_frac,n_bad_gt_segments,n_nonspeech_segments,n_nonspeech_tags,lang_script,lang_hint,latin_char_frac
0,0AEEA8NyVwY__11_609,0AEEA8NyVwY,https://www.youtube.com/watch?v=0AEEA8NyVwY,11.0,609.0,598.0,49,3,"Speaker A,Speaker B,Speaker C",1.29,...,561.77,564.39,2.62,0.00438,0,0,0,Devanagari,hi_or_mr,0.3145
1,0SoItGfM_sY__7_88,0SoItGfM_sY,https://www.youtube.com/watch?v=0SoItGfM_sY,7.0,88.0,81.0,11,5,"Speaker A,Speaker B,Speaker C,Speaker D,Speaker E",2.35,...,71.79,71.79,0.00,0.00000,0,0,0,Kannada,kn,0.2620
2,0VEwL9XZ0LY__261_557,0VEwL9XZ0LY,https://www.youtube.com/watch?v=0VEwL9XZ0LY,261.0,557.0,296.0,90,2,"Speaker A,Speaker B",0.07,...,280.80,290.72,9.92,0.03351,0,1,1,Devanagari,hi_or_mr,0.3022


## 1.3 — Dataset profile

Recomputes every number quoted in Step 0 from the file itself and writes
`results/dataset_profile.json`. Run it after any change to the parser: if a
number here moves, the Step 0 notes are stale.

In [16]:
import json
profile = data.profile_dataset(clips, cfg)

print("clips              ", profile["n_rows"], "rows /", profile["n_unique_video_ids"], "videos")
print("segments           ", profile["n_segments"], f"({profile['n_dropped_segments']} dropped as malformed)")
print("audio              ", profile["clip_duration_sec"]["total_hours"], "h")
print("speakers per clip  ", profile["speakers"]["per_clip_distribution"])
print("overlap            ", f"{profile['overlap']['overlap_frac_of_corpus']:.2%} of corpus time,",
      profile["overlap"]["n_clips_without_overlap"], "clips with none")
print("GT beyond end_sec  ", profile["gt_boundary"]["n_clips_gt_beyond_end_sec"], "clips,",
      f"median +{profile['gt_boundary']['overrun_median_sec']}s,",
      f"max +{profile['gt_boundary']['overrun_max_sec']}s")
print("scripts            ", profile["language"]["script_distribution"])
print("non-speech tags    ", profile["transcript"]["nonspeech_tags"])
print("code-switch glosses", profile["transcript"]["n_code_switch_glosses"],
      "in", profile["transcript"]["n_segments_with_gloss"], "segments")

print("\nDropped ground-truth segments:")
print(json.dumps(profile["dropped_segments"], indent=2, ensure_ascii=False))

16:47:23 | INFO    | sarvam_diar | dataset profile -> /content/drive/MyDrive/sarvam_diarization/results/dataset_profile.json
clips               100 rows / 99 videos
segments            9940 (2 dropped as malformed)
audio               12.276 h
speakers per clip   {2: 26, 3: 29, 4: 28, 5: 9, 6: 4, 7: 2, 8: 2}
overlap             7.16% of corpus time, 9 clips with none
GT beyond end_sec   85 clips, median +1.425s, max +6.41s
scripts             {'Devanagari': 25, 'Gujarati': 12, 'Telugu': 12, 'Tamil': 10, 'Kannada': 9, 'Oriya': 9, 'Bengali': 8, 'Gurmukhi': 8, 'Malayalam': 7}
non-speech tags     {'<unintelligible>': 264, '<noise>': 166, '<laughter>': 148, '<vocalization>': 81, '<background_speech>': 12, '<uhhh>': 1}
code-switch glosses 22560 in 4627 segments

Dropped ground-truth segments:
[
  {
    "clip_id": "6ZeRgvDHwcI__6_100",
    "index": 0,
    "speaker": "Speaker A",
    "start": 1.61,
    "end": 1.61,
    "text": "<noise>",
    "reason": "end <= start"
  },
  {
    "clip_id": "Q

## 1.4 — Extract

Per clip: resolve the media URL, range-fetch only `[start_sec, end_sec]` with
ffmpeg, force the exact sample count, publish atomically to Drive, then write
the sidecar as the commit marker.

**Fetch strategy.** Which YouTube player client yields a *downloadable* format
is not stable — it varies by network, video and yt-dlp version, and the client
with the best formats is often not a working one. Measured while building this:
yt-dlp's default rotation offers opus audio-only (itag 251) but those URLs 403
without a PO token, while `android` offers only itag 18 (muxed 360p) and
downloads fine. So the ladder is *walked*, not guessed, and whichever client
works is reused for the remaining clips — only the first clip pays for the
search. Tier A range-fetches just the window; tier B downloads the full track
and trims locally, and is used when tier A fails or its output drifts beyond
`DUR_TOL_SEC`. Both were verified to produce **byte-identical** audio.

Safe to interrupt and re-run — finished clips are skipped. If Colab hits
YouTube's bot gate, drop a Netscape-format `cookies.txt` into `<root>/data/`.

In [17]:
if flags.run_extraction:
    results = extraction.run(cfg, clips, flags)
else:
    results = pd.read_csv(cfg.extraction_csv)
    print(f"extraction skipped; loaded {len(results)} rows from {cfg.extraction_csv}")

16:47:23 | INFO    | sarvam_diar | step 1: 100 clip(s) selected of 100
16:47:23 | INFO    | sarvam_diar | [1/100] 0AEEA8NyVwY__11_609 extracting (no sidecar)
16:47:26 | WARNING | sarvam_diar | 0AEEA8NyVwY__11_609 attempt 1 (tier A, client default) failed [bot_check]
16:47:34 | WARNING | sarvam_diar | 0AEEA8NyVwY__11_609 attempt 2 (tier A, client android) failed [bot_check]
16:47:52 | WARNING | sarvam_diar | 0AEEA8NyVwY__11_609 attempt 3 (tier A, client web_safari,tv,ios,mweb) failed [bot_check]
16:48:12 | WARNING | sarvam_diar | 0AEEA8NyVwY__11_609 attempt 4 (tier B, client default) failed [bot_check]
16:48:12 | INFO    | sarvam_diar | [2/100] 0SoItGfM_sY__7_88 extracting (no sidecar)
16:48:15 | WARNING | sarvam_diar | 0SoItGfM_sY__7_88 attempt 1 (tier A, client default) failed [bot_check]
16:48:23 | WARNING | sarvam_diar | 0SoItGfM_sY__7_88 attempt 2 (tier A, client android) failed [bot_check]
16:48:41 | WARNING | sarvam_diar | 0SoItGfM_sY__7_88 attempt 3 (tier A, client web_safari,tv

KeyboardInterrupt: 

## 1.5 — Report

Per-video results, the run summary, and the failure breakdown. `all_exact` is
the one that matters: it asserts every published WAV holds exactly the requested
number of samples.

In [ ]:
import json

summary = json.load(open(cfg.extraction_summary))
print(json.dumps({k: summary[k] for k in
                  ("totals", "audio_contract", "download_tiers", "player_clients",
                   "formats", "error_classes", "session_counts")},
                 indent=2, ensure_ascii=False))

assert summary["audio_contract"]["all_exact"], "some WAVs are not exactly the requested length"

done = results[results.status == "ok"]
print(f"\n{len(done)} extracted / {len(results)} rows | "
      f"{summary['totals']['audio_hours']} h | {summary['totals']['bytes_human']}")

display(done[["clip_id", "requested_dur_sec", "n_samples", "n_expected_samples",
              "raw_delta_sec", "pad_samples", "trim_samples", "download_tier",
              "player_client", "ytdlp_format_id", "attempts", "elapsed_sec",
              "n_gt_speakers", "gt_overlap_frac", "ref_lang_hint"]].head(20))

failed = results[results.status == "failed"]
if len(failed):
    print(f"\n{len(failed)} failed ({failed.permanent.sum()} permanent):")
    display(failed[["clip_id", "error_class", "permanent", "attempts", "error_msg"]])
else:
    print("\nno failures")

# Every failed attempt, including ones that later succeeded on another rung.
attempts_log = pd.DataFrame(utils.read_jsonl(cfg.failures_jsonl))
if len(attempts_log):
    print(f"\nfailed attempts by class and client ({len(attempts_log)} total):")
    display(attempts_log.groupby(["error_class", "player_client"]).size()
            .rename("n").reset_index())

## 1.6 — Verification

Three checks, in order of how much they would cost to get wrong:

1. **Audit** — re-probes every published WAV against the exact-length contract
   and reports orphans, so a corrupted checkpoint is caught before Step 2 reads
   it as truth.
2. **Alignment** — the important one. Ground-truth timestamps are relative to
   `start_sec`, so a one-second trim offset would silently wreck DER for that
   clip with no visible symptom. Slices the first few ground-truth segments out
   of a published WAV and renders them next to their reference text: if the
   offset is wrong, the audio will not match the words.
3. **Idempotency** — re-running the extract cell must skip everything and touch
   the network zero times.

In [ ]:
# --- 1. audit every published WAV -------------------------------------------
audit = extraction.audit(cfg, clips)
attempted = audit[audit.clip_id.isin(results[results.status == "ok"].clip_id)]
print(f"audit: {attempted.valid.sum()}/{len(attempted)} extracted clips pass the exact-length contract")
bad = attempted[~attempted.valid]
if len(bad):
    display(bad)

orphans = {p.stem for p in cfg.audio_dir.glob("*.wav")} - {c.clip_id for c in clips}
print("orphan WAVs on Drive:", sorted(orphans) or "none")

In [ ]:
# --- 2. alignment spot-check ------------------------------------------------
# If the trim offset were wrong, the words below would not match the audio.
from IPython.display import Audio, display as ipy_display

ok_ids = results[results.status == "ok"].clip_id.tolist()
assert ok_ids, "extract at least one clip first"
clip = next(c for c in clips if c.clip_id == ok_ids[0])
wav = cfg.wav_path(clip.clip_id)

print(f"{clip.clip_id}  |  {clip.stats['lang_script']}  |  "
      f"{clip.stats['n_gt_speakers']} speakers  |  {len(clip.segments)} segments")

for seg in clip.segments[:4]:
    print(f"\n[{seg.start:7.2f} - {seg.end:7.2f}]  {seg.speaker}")
    print(f"   {seg.text[:160]}")
    ipy_display(Audio(extraction.read_wav_window(wav, seg.start, seg.end), rate=cfg.sample_rate))

In [ ]:
# --- 3. idempotency ---------------------------------------------------------
# Must report every already-extracted clip as skipped, with no network traffic.
_ = extraction.run(cfg, clips, StageFlags(only_clip_ids=ok_ids[:3]))

---

# Step 1.7 — Build the scoring reference

Everything so far has been pipeline-side. This section builds the **scoring
reference** from the ground truth, and puts a hard wall between the two.

The brief is unambiguous:

> **The ground truth is never an input to your pipeline.** `diarization_segments`
> and `asr_segments` are only for computing the final scores.

### Why the reference has to be built rather than scored raw

| Problem | Scale | Consequence if ignored |
|---|---|---|
| GT runs past the audio window | 85/100 clips, median +1.43 s | hypothesis silence scored against speech that is not in the file |
| Same-speaker intervals overlap themselves | 140 pairs | one speaker's second counted twice in the DER denominator |
| Corrupt segments | 2 (`end < start`, zero-length) | undefined turn geometry |
| **Code-switch gloss is not speech** | **24,914 of 149,121 tokens — 16.7%** | **a ~17% cpWER floor that says nothing about ASR quality** |

### What is deliberately *not* done

No overlap removal, no forgiveness collar as the headline metric, no
minimum-duration filter, no per-system normalization. Overlap in particular is
scored in full — the brief requires it.

### The line between normalization and cheating

The normalizer is legitimate because it is (1) a pure function of one text
string with no access to the other side, (2) applied **identically** to
reference and hypothesis, (3) fixed and versioned before any system output was
seen, and (4) documented with its corpus-wide token impact. `strip_gloss` is the
only asymmetry and it is inert — hypotheses contain no glosses, which the
verification cell asserts.

In [ ]:
# Ground truth in -> ClipInput (pipeline-safe) + ClipReference (scoring only).
inputs, ref_clips = data.split_reference(clips, results)
print(f"{len(inputs)} pipeline inputs (extracted clips only), {len(ref_clips)} reference clips")
print("ClipInput fields:", sorted(data.ClipInput.__dataclass_fields__))

if flags.build_reference:
    manifest = reference.run(cfg, clips)
else:
    manifest = pd.read_csv(cfg.reference_manifest)

display(manifest[["clip_id", "n_turns", "n_speakers", "speaker_time_sec", "overlap_sec",
                  "overlap_frac", "n_utterances", "n_ref_tokens",
                  "n_segments_dropped_by_uem", "n_segments_truncated_by_uem",
                  "n_same_speaker_merges"]].head(10))

print(f"\nturns {int(manifest.n_turns.sum())} | utterances {int(manifest.n_utterances.sum())} "
      f"| reference tokens {int(manifest.n_ref_tokens.sum())}")
print(f"UEM crop: {int(manifest.n_segments_dropped_by_uem.sum())} segments dropped, "
      f"{int(manifest.n_segments_truncated_by_uem.sum())} truncated, "
      f"{int(manifest.n_same_speaker_merges.sum())} same-speaker merges")
print("reference speaker counts:", manifest.n_speakers.value_counts().sort_index().to_dict())

In [ ]:
# --- normalization report + the golden table ---------------------------------
import json
report = reference.normalization_report(clips, cfg)
print(json.dumps(report, indent=2, ensure_ascii=False))

print("\nGolden cases -- every hard variant found in the corpus, eyeball these:\n")
for case in reference.GOLDEN_CASES:
    print("RAW :", case)
    print("NORM:", reference.normalize_text(case) or "(empty -- non-speech, excluded from WER)")
    print()

## 1.8 — Reference and leak-guard verification

Fifteen assertions. The ones that matter most:

* **Cropping must not change the speaker set** — otherwise speaker-count
  accuracy is measured against a reference the audio cannot support.
* **Cross-speaker overlap must survive at 7.13%** — a regression here means the
  reference quietly stopped scoring the overlap the brief requires.
* **The normalizer must be idempotent** and must never empty a speech segment.
* **`ClipInput` must carry no ground-truth field**, and the DataFrame guard must
  reject `n_gt_speakers` / `ref_lang_hint`. That column pair is exactly what
  would otherwise become pyannote's `num_speakers` or Whisper's `language`.

In [ ]:
import re, collections
refs = {c.clip_id: reference.build_reference(c) for c in clips}
speech = [s for c in clips for s in c.segments if s.is_speech]
failures = []

def check(name, ok, detail=""):
    print(f"  {'PASS' if ok else 'FAIL'}  {name}{'  ' + detail if detail else ''}")
    if not ok:
        failures.append(name)

print("=== reference integrity ===")
check("no turn outside the UEM",
      not [t for r in refs.values() for t in r.turns if t.end > r.uem[1] + 1e-6 or t.start < -1e-9])
check("cropping preserves the speaker set",
      all({s.speaker for s in c.segments} == {t.speaker for t in refs[c.clip_id].turns} for c in clips),
      f"{len(clips)}/{len(clips)}")
disjoint = True
for r in refs.values():
    per = collections.defaultdict(list)
    for t in r.turns:
        per[t.speaker].append((t.start, t.end))
    for v in per.values():
        v.sort()
        disjoint &= all(v[i][0] >= v[i - 1][1] - 1e-9 for i in range(1, len(v)))
check("same-speaker intervals disjoint after union", disjoint)
frac = sum(r.stats["overlap_sec"] for r in refs.values()) / sum(c.duration for c in clips)
check("cross-speaker overlap preserved", abs(frac - 0.0713) < 0.002, f"{frac:.4%} of corpus")
check("RTTM round-trips exactly",
      all(reference.parse_rttm(reference.to_rttm(r)) ==
          [data.Turn(t.speaker, round(t.start, 6), round(t.end, 6)) for t in r.turns]
          for r in refs.values()))

print("\n=== normalizer ===")
norm = [reference.normalize_text(s.text) for s in speech]
check("idempotent", all(reference.normalize_text(n) == n for n in norm), f"{len(speech)} segments")
check("no speech segment normalizes to empty", all(norm))
check("token count matches the report",
      sum(len(n.split()) for n in norm) == report["tokens"]["gloss_stripped_final"],
      f"{sum(len(n.split()) for n in norm)}")
check("reference is essentially pure native script",
      len(re.findall(r'[a-z]+', ' '.join(norm))) <= 40,
      f"{len(re.findall(r'[a-z]+', ' '.join(norm)))} residual Latin tokens")
gloss_free = ["नमस्कार मी गौरव जोशी आहे", "இது ஒரு சோதனை", "hello this is a test"]
check("gloss step is inert on gloss-free text (ref/hyp symmetry)",
      all(reference.normalize_text(h, True) == reference.normalize_text(h, False) for h in gloss_free))

print("\n=== leak guard ===")
fields = set(data.ClipInput.__dataclass_fields__)
check("ClipInput has no segments attribute", "segments" not in fields)
check("ClipInput carries no gt_/ref_ field", not utils.reference_fields(fields))
for bad in ("n_gt_speakers", "ref_lang_hint"):
    try:
        utils.assert_no_reference_fields(pd.DataFrame(columns=["clip_id", bad]))
        check(f"guard rejects {bad}", False)
    except AssertionError:
        check(f"guard rejects {bad}", True)
utils.assert_no_reference_fields(pd.DataFrame(columns=["clip_id", "wav_path"]))
check("guard passes clean columns", True)
check("split_reference yields ClipInput only",
      all(isinstance(v, data.ClipInput) for v in inputs.values()), f"{len(inputs)} inputs")

assert not failures, f"reference verification failed: {failures}"
print("\nALL CHECKS PASSED")